# Proyecto Final — Etapa 2: Clasificación de Textos por Década

**Objetivo:** predecir la década de origen (150–188) de un párrafo de texto histórico en español.

**Modelo principal:** `PlanTL-GOB-ES/roberta-base-bne` — RoBERTa entrenado sobre corpus histórico de la Biblioteca Nacional de España (BNE), lo que lo hace **especialmente adecuado** para este tipo de texto.

**Alternativa:** `dccuchile/bert-base-spanish-wwm-cased` (BETO) si el modelo BNE no está disponible.

**Estrategia:** fine-tuning completo con **AdamW**, warmup lineal, gradient clipping y checkpoint del mejor modelo según `val_acc`.

**Datos:** exclusivamente el conjunto de entrenamiento oficial. Sin uso de datos de prueba. Split 90/10 estratificado.

## 1. Instalación y librerías

In [ ]:
!pip install transformers accelerate -q

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.optim import AdamW

## 2. Configuración

In [ ]:
IS_KAGGLE = os.path.exists("/kaggle")
IS_COLAB  = 'COLAB_GPU' in os.environ

if IS_KAGGLE:
    DATA_DIR = "/kaggle/input/machine-learning-project"
    OUT_DIR  = "/kaggle/working"
elif IS_COLAB:
    # Montar Drive: from google.colab import drive; drive.mount('/content/drive')
    DATA_DIR = "/content/data"
    OUT_DIR  = "/content"
else:
    DATA_DIR = "../../data"
    OUT_DIR  = "../submissions"

TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
EVAL_PATH  = os.path.join(DATA_DIR, "eval.csv")
os.makedirs(OUT_DIR, exist_ok=True)

# RoBERTa entrenado sobre corpus histórico de la BNE — mejor opción para este problema
MODEL_NAME  = "PlanTL-GOB-ES/roberta-base-bne"
MAX_LEN     = 256
BATCH_SIZE  = 16
GRAD_ACCUM  = 2     # batch efectivo = 32
EPOCHS      = 5
LR          = 2e-5
WARMUP_FRAC = 0.1
SEED        = 42
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}  |  Mem: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 3. Datos

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
eval_df  = pd.read_csv(EVAL_PATH)

print(f"Train: {train_df.shape}  |  Eval: {eval_df.shape}")
print(f"Décadas: {sorted(train_df['decade'].unique())}")
train_df.head(3)

## 4. Preprocesamiento

**Label encoding** de 39 décadas. **Split 90/10 estratificado** para validación.

In [ ]:
le = LabelEncoder()
train_df["label"] = le.fit_transform(train_df["decade"])
NUM_CLASSES = len(le.classes_)
print(f"Clases: {NUM_CLASSES}")

train_data, val_data = train_test_split(
    train_df,
    test_size=0.1,
    random_state=SEED,
    stratify=train_df["label"],
)
print(f"Train: {len(train_data)}  |  Val: {len(val_data)}")

## 5. Tokenización y Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class DecadeDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts  = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True) if labels is not None else None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            str(self.texts[idx]),
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        item = {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
        }
        if self.labels is not None:
            item["labels"] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item


train_loader = DataLoader(
    DecadeDataset(train_data["text"], train_data["label"]),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    DecadeDataset(val_data["text"], val_data["label"]),
    batch_size=BATCH_SIZE * 2, num_workers=2, pin_memory=True,
)
eval_loader = DataLoader(
    DecadeDataset(eval_df["text"]),
    batch_size=BATCH_SIZE * 2, num_workers=2, pin_memory=True,
)
print("DataLoaders listos")

## 6. Modelo

**`PlanTL-GOB-ES/roberta-base-bne`** — RoBERTa preentrenado sobre más de 200 GB de texto español, incluyendo documentos históricos de la Biblioteca Nacional de España. Capa de clasificación lineal sobre el token `[CLS]` para 39 clases.

**Fuente del modelo:** https://huggingface.co/PlanTL-GOB-ES/roberta-base-bne  
**Licencia:** Apache 2.0

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
).to(DEVICE)

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parámetros totales:     {total_p:,}")
print(f"Parámetros entrenables: {trainable_p:,}")

## 7. Entrenamiento

**AdamW** con weight decay 0.01. **Cosine schedule con warmup** del 10%. **Gradient clipping** en 1.0. **Gradient accumulation** de 2 pasos (batch efectivo = 32). Se guarda el **mejor checkpoint** según `val_acc`.

In [ ]:
optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps  = (len(train_loader) // GRAD_ACCUM) * EPOCHS
scheduler    = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(WARMUP_FRAC * total_steps),
    num_training_steps=total_steps,
)


def train_epoch(loader):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    optimizer.zero_grad()
    for step, batch in enumerate(loader):
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        lbls = batch["labels"].to(DEVICE)
        out  = model(input_ids=ids, attention_mask=mask, labels=lbls)
        loss = out.loss / GRAD_ACCUM
        loss.backward()
        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        total_loss += out.loss.item()
        preds       = out.logits.argmax(-1)
        correct    += (preds == lbls).sum().item()
        n          += len(lbls)
    return total_loss / len(loader), correct / n


def eval_epoch(loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for batch in loader:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            lbls = batch["labels"].to(DEVICE)
            out  = model(input_ids=ids, attention_mask=mask, labels=lbls)
            total_loss += out.loss.item()
            preds       = out.logits.argmax(-1)
            correct    += (preds == lbls).sum().item()
            n          += len(lbls)
    return total_loss / len(loader), correct / n


best_val_acc = 0.0
best_ckpt    = os.path.join(OUT_DIR, "best_bne.pt")

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(train_loader)
    va_loss, va_acc = eval_epoch(val_loader)
    print(
        f"Epoch {epoch}/{EPOCHS}  "
        f"train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f}  "
        f"val_loss={va_loss:.4f}  val_acc={va_acc:.4f}"
    )
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), best_ckpt)
        print(f"  → Checkpoint guardado (val_acc={va_acc:.4f})")

print(f"\nMejor val_acc: {best_val_acc:.4f}")

## 8. Predicción y Envío

In [ ]:
model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
model.eval()

all_preds = []
with torch.no_grad():
    for batch in eval_loader:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        out  = model(input_ids=ids, attention_mask=mask)
        all_preds.extend(out.logits.argmax(-1).cpu().numpy())

submission = pd.DataFrame({
    "id":     eval_df["id"],
    "answer": le.inverse_transform(all_preds),
})

sub_path = os.path.join(OUT_DIR, "mark1.csv")
submission.to_csv(sub_path, index=False)
print(f"Submission guardado: {sub_path}")
print(f"\nDistribución:\n{submission['answer'].value_counts().sort_index()}")
submission.head(10)